# Becoming a BackProp Ninja :)

In [34]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [35]:
# read in all the words
words = open("names.txt", "r").read().splitlines()
print(len(words))

32033


In [36]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)

In [37]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [38]:
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [39]:
n_embd = 10 # dimensionality of the character embedding vectors
n_hidden = 64 # number of neurons in the hidden layer

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)

# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size) ** 0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1

# Layer 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1 
b2 = torch.randn(vocab_size, generator=g) * 0.1

#BatchNorm Parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

4137


In [40]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [41]:
# read in all the words
words = open("names.txt", "r").read().splitlines()
print(len(words))

32033


In [42]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)

In [43]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [44]:
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [45]:
n_embd = 10 # dimensionality of the character embedding vectors
n_hidden = 64 # number of neurons in the hidden layer

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)

# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size) ** 0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1

# Layer 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1 
b2 = torch.randn(vocab_size, generator=g) * 0.1

#BatchNorm Parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

4137


In [46]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] 

In [47]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] 

In [48]:
# Embedding the input
emb = C[Xb] # 32 training examples, 3 context characters for each example, and 10-number dimension for each character --> (32,3,10)
embcat = emb.view(emb.shape[0], -1) # flatten to become (32,30). Thus 32 training examples, each with 30-number rows (3 character * 10-number dimension per character). 

# Linear Layer 1
hprebn = embcat @ W1 + b1 # hidden layer's values before batchnorm is applied --> (32,30) @ (30,64) = (32,64)

# BatchNorm
bnmeani = 1/n * hprebn.sum(0, keepdim=True) # mean of each neuron across the batch
bndiff = hprebn - bnmeani # each value's deviation from it's neuron's mean
bndiff2 = bndiff**2 # bndiff squared
bnvar = 1/(n-1) * (bndiff2).sum(0, keepdim=True) # the variance of each neuron across the batch
bnvar_inv = (bnvar + 1e-5)**0.5 # the batchnorm variance but inverted 
bnraw = bndiff * bnvar_inv # each neuron now has mean zero and variance one across the batch (32,64)
hpreact = bngain * bnraw + bnbias # gain and bias used here to let the network rescale and reposition each neuron however it finds useful (that way batch norm does not force neurons to stay at mean zero and variance one). These are the hidden values after batchnorm but before non-linearity. 

# Nonlinearity
h = torch.tanh(hpreact) # hidden layer's values (activations) after nonlinearity.

# Linear Layer 2
logits = h @ W2 + b2 # (32,64) @ (64, 27) = (32,27). These are the raw, unnormalized output scores. 

# Manual Cross-Entropy Loss
logit_maxes = logits.max(1,keepdim=True).values # the largest logits in each row (32, 1)
norm_logits = logits - logit_maxes # purely for numerical stability in exp() later and it will not change the resulting probabilties
counts = norm_logits.exp() # unnormalized probabilties
counts_sum = counts.sum(1, keepdim=True) # sum across the 27 classes per example --> (32, 1)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv # actual softmax probabilities where each row sums to 1. (32, 27)
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean() # range(n) is [0...31] and Yb is the 32 correct next-character indices. logprobs[range(n), Yb] does some fancy indexing where, for row 0 it picks the column Yb[0] (e.g. for row 0 of logprobs it'll pick column 3 because Yb[0] = 3. It does this for every example and takes the average. The lower the better!)

# PyTorch Backward Pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, norm_logits, logit_maxes, logits, h, hpreact, bnraw, bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani, embcat, emb]:
    t.retain_grad()
loss.backward()
loss





tensor(3.5561, grad_fn=<NegBackward0>)

In [58]:
# backprop through the whole thing manually (through exactly all of the variables as defined in the forward pass above, one by one). 

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
dprobs = (1.0 / probs) * dlogprobs # local derivative of log(probs) is 1/probs.
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True) # in our forward pass, the (32,1) counts_sum_inv was expanded into (32,27) by copying it across columns. In the backward pass, we do the reverse by summing the (32,27) gradients back down to (32,1). 
dcounts = counts_sum_inv * dprobs
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = (counts) * dcounts
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits += F.one_hot(logits.max(1).indices, num_classes = logits.shape[1]) * dlogit_maxes

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)






cmp("logprobs", dlogprobs, logprobs)
cmp("probs", dprobs, probs)
cmp("counts_sum_inv", dcounts_sum_inv, counts_sum_inv)
cmp("counts_sum", dcounts_sum, counts_sum)
cmp("counts", dcounts, counts)
cmp("norm_logits", dnorm_logits, norm_logits)
cmp("logit_maxes", dlogit_maxes, logit_maxes)
cmp("h", dh, h)
cmp("W2", dW2, W2)
cmp("db2", db2, b2)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
db2             | exact: True  | approximate: True  | maxdiff: 0.0
